# 01_data_load.ipynb
- GitHub에서 DKTC 데이터 직접 받아옴 

In [1]:
import pandas as pd
import os
import requests
from pathlib import Path

In [12]:
# 실제 경로: data/train.csv 파일 하나에 전부 있음
url = "https://raw.githubusercontent.com/tunib-ai/DKTC/main/data/train.csv"

train_df = pd.read_csv(url, header=None, names=["id", "label_name", "conversation"])

print(train_df.shape)
print(train_df["label_name"].value_counts())

(3951, 3)
label_name
기타 괴롭힘 대화      1094
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896
class             1
Name: count, dtype: int64


In [11]:
train_df.head()

,id,label_name,conversation
0,idx,class,conversation
1,0,협박 대화,지금 너 스스로를 죽여달라고 애원하는 것인가?\n 아닙니다. 죄송합니다.\n 죽을 ...
2,1,협박 대화,길동경찰서입니다.\n9시 40분 마트에 폭발물을 설치할거다.\n네?\n똑바로 들어 ...
3,2,기타 괴롭힘 대화,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어.\n그만해. 니들 놀리는거 재미...
4,3,갈취 대화,어이 거기\n예??\n너 말이야 너. 이리 오라고\n무슨 일.\n너 옷 좋아보인다?...


In [13]:
# 라벨 매핑 (실제 컬럼값 기준)
# "라벨 이름(문자열)" → "라벨 번호(정수)"로 대응표 딕셔너리 
label_map = {
    "협박 대화": 0,
    "갈취 대화": 1,
    "직장 내 괴롭힘 대화": 2,
    "기타 괴롭힘 대화": 3
}

# 매핑
train_df["label"] = train_df["label_name"].map(label_map)

# 매핑 확인
print(train_df["label"].value_counts())
print(train_df["label_name"].value_counts())
print(train_df[train_df["label"].isna()])  # 매핑 안 된 행 확인

label
3.0    1094
1.0     981
2.0     979
0.0     896
Name: count, dtype: int64
label_name
기타 괴롭힘 대화      1094
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896
class             1
Name: count, dtype: int64
    id label_name  conversation  label
0  idx      class  conversation    NaN


In [14]:
train_df.head()

,id,label_name,conversation,label
0,idx,class,conversation,NaN
1,0,협박 대화,지금 너 스스로를 죽여달라고 애원하는 것인가?\n 아닙니다. 죄송합니다.\n 죽을 ...,0.0
2,1,협박 대화,길동경찰서입니다.\n9시 40분 마트에 폭발물을 설치할거다.\n네?\n똑바로 들어 ...,0.0
3,2,기타 괴롭힘 대화,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어.\n그만해. 니들 놀리는거 재미...,3.0
4,3,갈취 대화,어이 거기\n예??\n너 말이야 너. 이리 오라고\n무슨 일.\n너 옷 좋아보인다?...,1.0


In [ ]:
# 헤더 행 제거
train_df = train_df[train_df["id"] != "idx"].reset_index(drop=True)
#  타입 정리
train_df["label"] = train_df["label"].astype(int)
train_df["id"] = train_df["id"].astype(int)

print(f"shape: {train_df.shape}")
print(train_df["label_name"].value_counts())
print(train_df.dtypes)

shape: (3950, 4)
label_name
기타 괴롭힘 대화      1094
갈취 대화           981
직장 내 괴롭힘 대화     979
협박 대화           896
Name: count, dtype: int64
id               int64
label_name      object
conversation    object
label            int64
dtype: object


In [16]:
train_df.head()

,id,label_name,conversation,label
0,0,협박 대화,지금 너 스스로를 죽여달라고 애원하는 것인가?\n 아닙니다. 죄송합니다.\n 죽을 ...,0
1,1,협박 대화,길동경찰서입니다.\n9시 40분 마트에 폭발물을 설치할거다.\n네?\n똑바로 들어 ...,0
2,2,기타 괴롭힘 대화,너 되게 귀여운거 알지? 나보다 작은 남자는 첨봤어.\n그만해. 니들 놀리는거 재미...,3
3,3,갈취 대화,어이 거기\n예??\n너 말이야 너. 이리 오라고\n무슨 일.\n너 옷 좋아보인다?...,1
4,4,갈취 대화,저기요 혹시 날이 너무 뜨겁잖아요? 저희 회사에서 이 선크림 파는데 한 번 손등에 ...,1


In [5]:
# 파일저장
train_df.to_csv("data/train.csv", index=False)
print("저장 완료")

저장 완료


# 02_model_train

In [17]:
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from datasets import Dataset

# GPU 확인 (로컬용)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"디바이스: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")



/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


디바이스: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


In [18]:
# 데이터 로드 및 분할
df = pd.read_csv("data/train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)
print(f"학습: {len(train_df)}, 검증: {len(val_df)}")
print(train_df["label_name"].value_counts())

학습: 3160, 검증: 790
label_name
기타 괴롭힘 대화      875
갈취 대화          785
직장 내 괴롭힘 대화    783
협박 대화          717
Name: count, dtype: int64


In [8]:
# 토크나이저 로드
MODEL_NAME = "klue/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("토크나이저 로드 완료")

# 토크나이징
def tokenize(batch):
    return tokenizer(
        batch["conversation"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = Dataset.from_pandas(train_df[["conversation", "label"]].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[["conversation", "label"]].reset_index(drop=True))

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"학습 데이터셋: {train_dataset}")
print(f"검증 데이터셋: {val_dataset}")

토크나이저 로드 완료


Map: 100%|██████████| 790/790 [00:00<00:00, 15640.93 examples/s]

학습 데이터셋: Dataset({
    features: ['conversation', 'label', 'input_ids', 'attention_mask'],
    num_rows: 3160
})
검증 데이터셋: Dataset({
    features: ['conversation', 'label', 'input_ids', 'attention_mask'],
    num_rows: 790
})


In [9]:
# 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4
)
model.to(device)
print("모델 로드 완료")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1844.90it/s]
RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 로드 완료


In [10]:
# 평가 함수
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="macro")
    return {"f1_macro": f1}

# 학습 설정
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=True,
    report_to="none"   # wandb 연동 끄기
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# 학습 시작
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.845078,0.385773,0.878412
2,0.300175,0.405372,0.858893
3,0.165924,0.433709,0.879295
4,0.076296,0.493336,0.875249
5,0.051218,0.443279,0.891986


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.74s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=495, training_loss=0.2480860568056203, metrics={'train_runtime': 79.6897, 'train_samples_per_second': 198.269, 'train_steps_per_second': 6.212, 'total_flos': 1039307331379200.0, 'train_loss': 0.2480860568056203, 'epoch': 5.0})

In [11]:
# 최종 평가
label_names = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘"]

preds = trainer.predict(val_dataset)
pred_labels = preds.predictions.argmax(-1)
true_labels = val_df["label"].values

print(classification_report(true_labels, pred_labels, target_names=label_names))

              precision    recall  f1-score   support

          협박       0.87      0.83      0.85       179
          갈취       0.86      0.91      0.89       196
      직장내괴롭힘       0.96      0.96      0.96       196
       기타괴롭힘       0.88      0.86      0.87       219

    accuracy                           0.89       790
   macro avg       0.89      0.89      0.89       790
weighted avg       0.89      0.89      0.89       790



# Threshold 개선

In [12]:
import numpy as np
import torch.nn.functional as F
import torch

# 검증셋으로 threshold 실험
THRESHOLD = 0.6  # 이 값 조정하면서 테스트

preds_output = trainer.predict(val_dataset)
logits = torch.tensor(preds_output.predictions)
probs = F.softmax(logits, dim=-1).numpy()

# 최대 확률이 threshold 미만이면 → 일반대화(4)로 분류
pred_labels_threshold = []
for prob in probs:
    if prob.max() < THRESHOLD:
        pred_labels_threshold.append(4)  # 일반대화
    else:
        pred_labels_threshold.append(prob.argmax())

pred_labels_threshold = np.array(pred_labels_threshold)

# 현재 검증셋엔 일반대화가 없으니까 4로 분류된 것만 확인
n_normal = (pred_labels_threshold == 4).sum()
print(f"일반대화로 분류된 수: {n_normal} / {len(pred_labels_threshold)}")
print(f"threshold {THRESHOLD} 기준 accuracy: {(pred_labels_threshold[:len(true_labels)] == true_labels).mean():.4f}")



# threshold 0.6에서 790개 중 5개만 일반대화로 분류됨 
# 모델이 거의 모든 대화에 대해 과하게 확신하고 있다는 뜻.
# threshold를 올려서 실험해야 함

일반대화로 분류된 수: 13 / 790
threshold 0.6 기준 accuracy: 0.8861


In [13]:
# 여러 threshold 한번에 실험
thresholds = [0.6, 0.7, 0.8, 0.9, 0.95]

for t in thresholds:
    pred_t = []
    for prob in probs:
        if prob.max() < t:
            pred_t.append(4)
        else:
            pred_t.append(prob.argmax())
    pred_t = np.array(pred_t)
    
    n_normal = (pred_t == 4).sum()
    acc = (pred_t[pred_t != 4] == true_labels[pred_t != 4]).mean()
    print(f"threshold {t} | 일반대화 판정: {n_normal:3d}개 | 나머지 accuracy: {acc:.4f}")

threshold 0.6 | 일반대화 판정:  13개 | 나머지 accuracy: 0.9009
threshold 0.7 | 일반대화 판정:  19개 | 나머지 accuracy: 0.9027
threshold 0.8 | 일반대화 판정:  29개 | 나머지 accuracy: 0.9080
threshold 0.9 | 일반대화 판정:  59개 | 나머지 accuracy: 0.9207
threshold 0.95 | 일반대화 판정:  71개 | 나머지 accuracy: 0.9277


In [14]:
# 일반 대화 샘플 직접 넣어서 확률 확인
test_sentences = [
    "오늘 점심 뭐 먹었어? 나는 김치찌개 먹었는데 진짜 맛있었어",
    "주말에 영화 볼래? 요즘 재밌는 거 많이 나왔던데",
    "야 어제 그 드라마 봤어? 진짜 반전 미쳤다",
    "나 요즘 너무 피곤해 야근이 너무 많아",
    "내일 날씨 어때? 우산 챙겨야 하나",
]

label_names = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]

model.eval()
for sent in test_sentences:
    inputs = tokenizer(
        sent,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
        prob = F.softmax(logits, dim=-1).cpu().numpy()[0]
    
    max_prob = prob.max()
    pred_class = label_names[prob.argmax()] if max_prob >= 0.9 else "일반대화"
    
    print(f"입력: {sent[:30]}...")
    print(f"예측: {pred_class} | 최대확률: {max_prob:.4f} | 분포: {prob.round(3)}")
    print()

# "김치찌개 먹었어"를 직장내괴롭힘 0.997로 확신하는 모델
# threshold를 아무리 올려도 이런 케이스는 못 걸러냄
# 모델이 일반 대화를 아예 본 적이 없어서 생기는 구조적 문제.

입력: 오늘 점심 뭐 먹었어? 나는 김치찌개 먹었는데 진짜 맛...
예측: 직장내괴롭힘 | 최대확률: 0.9874 | 분포: [0.005 0.002 0.987 0.006]

입력: 주말에 영화 볼래? 요즘 재밌는 거 많이 나왔던데...
예측: 일반대화 | 최대확률: 0.5923 | 분포: [0.071 0.015 0.321 0.592]

입력: 야 어제 그 드라마 봤어? 진짜 반전 미쳤다...
예측: 기타괴롭힘 | 최대확률: 0.9930 | 분포: [0.005 0.001 0.001 0.993]

입력: 나 요즘 너무 피곤해 야근이 너무 많아...
예측: 직장내괴롭힘 | 최대확률: 0.9832 | 분포: [0.006 0.001 0.983 0.01 ]

입력: 내일 날씨 어때? 우산 챙겨야 하나...
예측: 기타괴롭힘 | 최대확률: 0.9877 | 분포: [0.005 0.005 0.002 0.988]



In [15]:
# 일반 대화 샘플 직접 생성 (few-shot용)
normal_conversations = [
    "오늘 점심 뭐 먹었어?\n김치찌개 먹었어 맛있었는데\n나도 먹고 싶다 어디야?\n회사 근처 식당인데 같이 가자",
    "주말에 뭐해?\n그냥 집에 있을 것 같아\n나랑 카페나 갈래?\n좋아 몇시에 볼까",
    "어제 드라마 봤어?\n응 진짜 재밌더라 반전 미쳤다\n나도 봤는데 그 장면에서 소름돋았잖아\n진짜 다음주가 기대된다",
    "요즘 날씨 너무 좋다\n맞아 산책하기 딱 좋은 날씨야\n우리 주말에 한강 갈까\n좋아 자전거 빌려서 타자",
    "나 요즘 운동 시작했어\n오 어떤 운동?\n헬스장 등록했어 일주일째 가고 있어\n대단하다 나도 같이 가도 돼?",
    "밥 먹었어?\n아직 못 먹었어 바빠서\n그러면 안되지 뭐라도 먹어\n알았어 이따 먹을게",
    "이번 주말 날씨 어때?\n비 온다던데 우산 챙겨\n진짜? 나 소풍 가려고 했는데\n다음 주말로 미루자",
    "점심 같이 먹을래?\n좋아 오늘 뭐 먹고 싶어?\n짜장면 어때?\n완전 좋아 빨리 가자",
    "오늘 퇴근하고 뭐해?\n그냥 집에서 쉬려고\n우리 오랜만에 저녁 먹자\n좋아 6시에 봐",
    "요즘 책 읽고 있어?\n응 소설 읽고 있는데 재밌어\n어떤 책이야?\n추천해줄게 나중에",
    "지난주에 여행 어땠어?\n완전 좋았어 날씨도 맑고\n사진 보여줘\n카톡으로 보내줄게",
    "아침에 커피 마셨어?\n응 요즘 아메리카노 없으면 못 살아\n나도 그래 카페인 중독인 것 같아\n우리 커피 마시러 가자",
    "오늘 많이 피곤해 보여\n어젯밤에 잠을 못 잤어\n왜? 무슨 일 있었어?\n그냥 잠이 안 오더라고",
    "저녁 뭐 먹을까?\n오늘은 치킨 먹고 싶다\n좋아 시켜먹자\n반반으로 주문하자",
    "오늘 회의 어땠어?\n생각보다 빨리 끝났어\n다행이다 나는 두 시간 걸렸어\n오래 했네 힘들었겠다",
]

# 데이터프레임으로 변환
normal_df = pd.DataFrame({
    "id": range(len(normal_conversations)),
    "label_name": "일반 대화",
    "conversation": normal_conversations,
    "label": 4
})

print(f"일반 대화 샘플: {len(normal_df)}개")
print(normal_df.head(3))

일반 대화 샘플: 15개
   id label_name                                       conversation  label
0   0      일반 대화  오늘 점심 뭐 먹었어?\n김치찌개 먹었어 맛있었는데\n나도 먹고 싶다 어디야?\n회...      4
1   1      일반 대화      주말에 뭐해?\n그냥 집에 있을 것 같아\n나랑 카페나 갈래?\n좋아 몇시에 볼까      4
2   2      일반 대화  어제 드라마 봤어?\n응 진짜 재밌더라 반전 미쳤다\n나도 봤는데 그 장면에서 소름...      4


In [16]:
# 기존 데이터와 합치기
combined_df = pd.concat([train_df, normal_df], ignore_index=True)
print(f"\n전체: {len(combined_df)}개")
print(combined_df["label_name"].value_counts())


전체: 3175개
label_name
기타 괴롭힘 대화      875
갈취 대화          785
직장 내 괴롭힘 대화    783
협박 대화          717
일반 대화           15
Name: count, dtype: int64


In [17]:
# 일반 대화 오버샘플링
normal_oversampled = normal_df.sample(n=200, replace=True, random_state=42)

combined_df = pd.concat([train_df, normal_oversampled], ignore_index=True)
print(combined_df["label_name"].value_counts())

label_name
기타 괴롭힘 대화      875
갈취 대화          785
직장 내 괴롭힘 대화    783
협박 대화          717
일반 대화          200
Name: count, dtype: int64


In [18]:
# train/val 분할 (일반 대화 포함)
from sklearn.model_selection import train_test_split

train_df2, val_df2 = train_test_split(
    combined_df,
    test_size=0.2,
    stratify=combined_df["label"],
    random_state=42
)
print(f"학습: {len(train_df2)}, 검증: {len(val_df2)}")
print(train_df2["label_name"].value_counts())

학습: 2688, 검증: 672
label_name
기타 괴롭힘 대화      700
갈취 대화          628
직장 내 괴롭힘 대화    626
협박 대화          574
일반 대화          160
Name: count, dtype: int64


In [19]:
# 5클래스로 토크나이징
train_dataset2 = Dataset.from_pandas(train_df2[["conversation", "label"]].reset_index(drop=True))
val_dataset2 = Dataset.from_pandas(val_df2[["conversation", "label"]].reset_index(drop=True))

train_dataset2 = train_dataset2.map(tokenize, batched=True)
val_dataset2 = val_dataset2.map(tokenize, batched=True)

train_dataset2.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset2.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print("토크나이징 완료")

Map: 100%|██████████| 672/672 [00:00<00:00, 15889.66 examples/s]

토크나이징 완료


In [20]:
# 5클래스 모델 새로 로드
model2 = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5
)
model2.to(device)

# class weight 계산
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2,3,4]),
    y=train_df2["label"].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"class weights: {class_weights.round(3)}")

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1857.36it/s]
RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


class weights: [0.937 0.856 0.859 0.768 3.36 ]


In [21]:
# weighted loss 적용한 커스텀 Trainer
from torch.nn import CrossEntropyLoss

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = CrossEntropyLoss(weight=class_weights_tensor)(logits, labels)
        return (loss, outputs) if return_outputs else loss

training_args2 = TrainingArguments(
    output_dir="./results_5class",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    fp16=True,
    report_to="none"
)

trainer2 = WeightedTrainer(
    model=model2,
    args=training_args2,
    train_dataset=train_dataset2,
    eval_dataset=val_dataset2,
    compute_metrics=compute_metrics,
)

trainer2.train()
# Epoch 3이 최적점. 4클래스 모델(0.895)보다 F1 0.909로 향상

Epoch,Training Loss,Validation Loss,F1 Macro
1,0.710094,0.332537,0.892507
2,0.223170,0.325030,0.887409
3,0.127925,0.374540,0.909118
4,0.061470,0.411701,0.905314
5,0.028206,0.419879,0.908539


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.74s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=420, training_loss=0.20364202637047996, metrics={'train_runtime': 69.4505, 'train_samples_per_second': 193.519, 'train_steps_per_second': 6.047, 'total_flos': 884076958679040.0, 'train_loss': 0.20364202637047996, 'epoch': 5.0})

In [22]:
label_names_5 = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]

preds2 = trainer2.predict(val_dataset2)
pred_labels2 = preds2.predictions.argmax(-1)
true_labels2 = val_df2["label"].values

print(classification_report(true_labels2, pred_labels2, target_names=label_names_5))

              precision    recall  f1-score   support

          협박       0.91      0.83      0.86       143
          갈취       0.86      0.83      0.84       157
      직장내괴롭힘       0.93      0.97      0.95       157
       기타괴롭힘       0.86      0.91      0.89       175
        일반대화       1.00      1.00      1.00        40

    accuracy                           0.89       672
   macro avg       0.91      0.91      0.91       672
weighted avg       0.89      0.89      0.89       672



In [23]:
# 아까 실패했던 일반 대화 문장 다시 테스트
model2.eval()
for sent in test_sentences:
    inputs = tokenizer(
        sent,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        logits = model2(**inputs).logits
        prob = F.softmax(logits, dim=-1).cpu().numpy()[0]
    
    pred_class = label_names_5[prob.argmax()]
    print(f"입력: {sent[:25]}...")
    print(f"예측: {pred_class} | 확률: {prob.round(3)}")
    print()

입력: 오늘 점심 뭐 먹었어? 나는 김치찌개 먹었는데...
예측: 일반대화 | 확률: [0.001 0.001 0.001 0.001 0.997]

입력: 주말에 영화 볼래? 요즘 재밌는 거 많이 나왔...
예측: 일반대화 | 확률: [0.001 0.001 0.001 0.001 0.997]

입력: 야 어제 그 드라마 봤어? 진짜 반전 미쳤다...
예측: 일반대화 | 확률: [0.001 0.001 0.001 0.001 0.997]

입력: 나 요즘 너무 피곤해 야근이 너무 많아...
예측: 일반대화 | 확률: [0.001 0.    0.001 0.001 0.997]

입력: 내일 날씨 어때? 우산 챙겨야 하나...
예측: 일반대화 | 확률: [0.001 0.001 0.001 0.001 0.997]



In [24]:
# 모델 저장
save_path = "./models/dktc_5class"
trainer2.save_model(save_path)
tokenizer.save_pretrained(save_path)
print(f"저장 완료: {save_path}")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]

저장 완료: ./models/dktc_5class


# 추론 노트북 (03_inference.ipynb)

In [25]:
# 전체 추론 함수 완성본
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

LABEL_NAMES = ["협박", "갈취", "직장내괴롭힘", "기타괴롭힘", "일반대화"]
MODEL_PATH = "./models/dktc_5class"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def predict(text: str) -> dict:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()[0]
    
    pred_idx = probs.argmax()
    return {
        "label": LABEL_NAMES[pred_idx],
        "confidence": round(float(probs[pred_idx]), 4),
        "all_probs": {name: round(float(p), 4) for name, p in zip(LABEL_NAMES, probs)}
    }

# 테스트
test_cases = [
    "당장 돈 안 가져오면 죽여버린다",
    "오늘 회의 몇 시야?",
    "너 그 돈 언제 갚을 거야 진짜",
]

for text in test_cases:
    result = predict(text)
    print(f"입력: {text}")
    print(f"예측: {result['label']} ({result['confidence']*100:.1f}%)")
    print()

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1757.82it/s]


입력: 당장 돈 안 가져오면 죽여버린다
예측: 협박 (98.4%)

입력: 오늘 회의 몇 시야?
예측: 일반대화 (99.6%)

입력: 너 그 돈 언제 갚을 거야 진짜
예측: 협박 (98.6%)



# 오류 분석 (가장 가성비 높음)

In [ ]:
# 모델이 틀린 케이스만 뽑아서 패턴 찾기
wrong_idx = (pred_labels2 != true_labels2)
wrong_df = val_df2[wrong_idx].copy()
wrong_df["predicted"] = [label_names_5[p] for p in pred_labels2[wrong_idx]]
wrong_df.groupby(["label_name", "predicted"]).size()


# 패턴 1: 갈취 ↔ 협박 상호 혼동
# 갈취→협박 7건, 협박→갈취 13건
# 언어적으로 겹침: "안 하면 가만 안 둔다" 류의 문장이 두 클래스 모두에 등장
# 갈취는 금전/물질 요구가 명시돼야 하지만, 대화 초반엔 협박 언어가 먼저 나옴

# 패턴 2: 갈취 → 기타괴롭힘 (13건)
# 직접적 금전 요구 없이 압박하는 문장이 기타괴롭힘으로 분류됨
# 갈취의 전조 단계 대화가 기타괴롭힘과 구분 안 됨

# 패턴 3: 직장내괴롭힘은 거의 안 틀림 (4건)
# 도메인 특화 어휘("야근", "보고서", "팀장") 덕분에 구분이 쉬움


label_name   predicted
갈취 대화        기타괴롭힘        13
             직장내괴롭힘        7
             협박            7
기타 괴롭힘 대화    갈취            9
             직장내괴롭힘        3
             협박            3
직장 내 괴롭힘 대화  기타괴롭힘         2
             협박            2
협박 대화        갈취           13
             기타괴롭힘        11
             직장내괴롭힘        1
dtype: int64

In [29]:
# 실제 오류 문장 확인
# 갈취→기타괴롭힘 오분류 케이스 직접 확인
mask = (val_df2["label_name"] == "갈취 대화") & (wrong_df["predicted"] == "기타괴롭힘")
wrong_df[mask]["conversation"].head(3).values

/tmp/ipykernel_13184/3840484615.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  wrong_df[mask]["conversation"].head(3).values


array(['어이 꼬마.\n저. 저말인가요??\n그래 너말야.\n왜. 부르시나요?\n일로 와봐.\n알겠습니다. 근데 왜 부르셨나요?\n너 손목에 좋은거 차고 다니네?\n앗.\n좋은말로 할 때 내놔.\n네.',
       '결혼 축하해 해지야\n어 고마워\n아니 그런데 너는 결혼한다고 청첩장 돌리는 애가 서운하게 나한테만 연락안했더라\n아 우리 별로 그렇게 안친하다고 생각해서\n애 서운하게 나는 그렇게 생각안했어\n아 그래 미안해 와줘서 고마워\n근데 그거 알아 너네 신랑?\n뭐? 말하는거야\n너 가슴수술한거 그 전에 남자친구가 너 가슴 아스팔트껌딱지 같다 그래서 충격먹고 그해 여름방학에 수술한거\n아니 몰라\n음 니네 신랑도 너 수술 안한지 알텐데 알면 차이는거아니야?\n우리 신랑 그런사람 아니야\n그럼 내가 말해도 돼?\n왜 그러는거야?\n아니 나도 결혼하는데 좀 자금이 부족해서 너는 있는 집에 시집간다는데 친구 좋다는게 뭐야\n나좀 도와주라고 한 3천만 빌려주라 쓰고 줄게\n나 그정도 돈 없어\n그래? 니네 신랑 그 역 4번 출구 앞에서 회계사 한다고 했나? 니 친구라고 상담할거 있다고 한번 가볼까?\n빌려줄게',
       '밥한번만 사주라\n또?\n또라니\n저번에도 내가 샀잖아\n그건 다른이유고 오늘은 축하할일이 있으니깐 한번만\n다른사람한테 사달라하면안돼?\n너가 좋아하서 그렇지 친구좋다는게뭐냐\n.\n대신 너가 좋아하는 거 먹쟈\n.알겠어'],
      dtype=object)

In [30]:
# 협박→갈취 오분류 케이스
mask2 = (val_df2["label_name"] == "협박 대화") & (wrong_df["predicted"] == "갈취")
wrong_df[mask2]["conversation"].head(3).values

/tmp/ipykernel_13184/2206184973.py:3: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  wrong_df[mask2]["conversation"].head(3).values


array(['너 왜 어제 연락이 없었어.\n어제 바빠서 그랬어. 왜 또 그래?\n내가 분명 말했지. 내가 전화 걸면 바로 받으라고.\n사람이 바쁘면 신경 못 쓸 수도 있지.\n이런 일이 한 두번이니? 왜 저녁에 바빴는데?\n아 너랑 말 섞기 귀찮다. 해명하는 것도 이제 지쳐.\n너만 지쳐? 너 남자 만났지?\n그건 또 무슨 소리래. 말 걸지마.\n너 그 핸드폰 내놔. 손목 부시기 전에 내놔.\n왜 말을 그런 식으로 해? 싫어.',
       '많이 어려워?\n제가 그 물건을 다음달까지 어떻게.\n안 어렵게 도와줄까?\n네? 그게 무슨.\n이게 누구실까? \n앗. 우리 아들\n그래 하나뿐인 석박사 수석 아드님 \n왜 이러십니까. \n어때 좀 쉽게 구할 거 같아?\n제발.시간을 좀 주세요.',
       '너 바람폈지?\n아니 안폈는데\n나에게 증거사진이 있어\n무슨증거?\n자 이거봐 이거 너 맞잖아\n뭐야 몰래 촬영하는건 불법이야 내놔\n범죄자가 무슨 말이 많아.\n범죄자라니\n너 이거 주변사람들에게 다 퍼뜨릴꺼야 넌 이제 끝났어\n너 협박죄로 고소할거고 사진 어서 내놔'],
      dtype=object)

In [31]:
# Grad-CAM 대신 토큰 중요도 시각화
# ! pip install transformers_interpret 
from transformers_interpret import SequenceClassificationExplainer
explainer = SequenceClassificationExplainer(model, tokenizer)
word_attributions = explainer("당장 돈 안 가져오면 죽여버린다")
explainer.visualize()

True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,LABEL_0 (0.98),LABEL_0,2.03,[CLS] 당장 돈 안 가져오 ##면 죽여 ##버린 ##다 [SEP]


True Label,Predicted Label,Attribution Label,Attribution Score,Word Importance
0,LABEL_0 (0.98),LABEL_0,2.03,[CLS] 당장 돈 안 가져오 ##면 죽여 ##버린 ##다 [SEP]
